In [2]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from tenacity import retry, wait_exponential
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pydantic import BaseModel, Field

In [7]:
load_dotenv(override=True)

MODEL_LLM = "gpt-5-nano"
MODEL_EMBEDDING = "text-embedding-3-large"
DB_NAME = "vector_db"
RETRIEVAL_K = 20
FINAL_K =10
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [4]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [5]:
def rewrite_query(question: str, history: list[dict] = []) -> str:
    """
    Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base.

    Args:
        question (str): Question of the user
        history (list[dict]): Conversation history

    Returns:
        str: Rewritten question to get relevant content from the knowledge base
    """

    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a short, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
IMPORTANT: Respond ONLY with the precise knowledgebase query, nothing else.
"""
    messages = [SystemMessage(content=message)]
    llm = ChatOpenAI(temperature=0, model=MODEL_LLM)
    response = llm.invoke(input=messages)
    return response.content

In [13]:
original_question = "Who won the prestigious IIOTY award in 2023?"

In [14]:
rewritten_question = rewrite_query(question=original_question)

In [15]:
rewritten_question

'Who won IIOTY 2023?'

In [16]:
def fetch_content_unranked(question: str) -> list[Document]:
    """
    Retrieve relevant context documents for a question

    Args:
        question (str): Question from the user
    Returns:
        list[Document]: List of relevant documents from vector store
    """

    embeddings = OpenAIEmbeddings(model=MODEL_EMBEDDING)
    vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
    retriever = vectorstore.as_retriever()

    return retriever.invoke(input=question, k=RETRIEVAL_K)

In [17]:
chunks_original = fetch_content_unranked(question=original_question)

In [18]:
chunks_original

[Document(id='19dbb1bc-5bb4-4842-b4f4-276922aad7e9', metadata={'source': 'knowledge-base\\employees\\Maxine Thompson.md', 'doc_type': 'employees'}, page_content='- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.'),
 Document(id='6ef603d2-6a01-410c-b8e6-6422011597db', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content='- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  \n\n- **2023**: **Exceeds Expectations**  \n  Market leadership w

In [19]:
chunks_rewritten = fetch_content_unranked(question=rewritten_question)

In [20]:
chunks_rewritten

[Document(id='19dbb1bc-5bb4-4842-b4f4-276922aad7e9', metadata={'source': 'knowledge-base\\employees\\Maxine Thompson.md', 'doc_type': 'employees'}, page_content='- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.'),
 Document(id='6ef603d2-6a01-410c-b8e6-6422011597db', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content='- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  \n\n- **2023**: **Exceeds Expectations**  \n  Market leadership w

In [21]:
def merge_chunks(chunks_original: list[Document], chunks_rewritten: list[Document]) -> list[Document]:
    """
    Take two chunks and merge into one by removing duplicate

    Args:
        chunks_original (list[Document]): Relevant content chunks retrieved for original question of the user
        chunks_rewritten (list[Document]): Relevant content chunks retrieved for rewritten question

    Returns:
        list[Document]: Merged chunks after removing duplicate
    """

    chunks_merged = chunks_original[:]
    existing = [chunk.page_content for chunk in chunks_original]
    for chunk in chunks_rewritten:
        if chunk.page_content not in existing:
            chunks_merged.append(chunk)

    return chunks_merged

In [22]:
chunks_merged = merge_chunks(chunks_original=chunks_original, chunks_rewritten=chunks_rewritten)

In [23]:
chunks_merged

[Document(id='19dbb1bc-5bb4-4842-b4f4-276922aad7e9', metadata={'source': 'knowledge-base\\employees\\Maxine Thompson.md', 'doc_type': 'employees'}, page_content='- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.'),
 Document(id='6ef603d2-6a01-410c-b8e6-6422011597db', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content='- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing employee concerns, leading to overall improvement despite a saturated market.  \n\n- **2023**: **Exceeds Expectations**  \n  Market leadership w

In [28]:
def rerank(question: str, chunks: list[Document]) -> list[Document]:
    """
    Rerank the list of relevant content according to the user question

    Args:
        question (str): Question of the user
        chunks (list[Document]): Relevant content chunks to be reranked

    Returns:
        list[Document]: Reranked relevant content chunks
    """

    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."

    messages = [SystemMessage(content=system_prompt)]
    messages.append(HumanMessage(content=user_prompt))
    llm = ChatOpenAI(temperature=0, model=MODEL_LLM).with_structured_output(schema=RankOrder)
    response = llm.invoke(input=messages)

    order = response.order
    return [chunks[i - 1] for i in order]

In [29]:
chunks_reranked = rerank(question=original_question, chunks=chunks_merged)

In [30]:
chunks_reranked

[Document(id='19dbb1bc-5bb4-4842-b4f4-276922aad7e9', metadata={'source': 'knowledge-base\\employees\\Maxine Thompson.md', 'doc_type': 'employees'}, page_content='- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.'),
 Document(id='02d3d5f4-de51-403c-8a26-3f6dcba4cc1c', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Maxine Thompson.md'}, page_content='## Compensation History\n- **2017**: $70,000 (Junior Data Engineer)  \n- **2018**: $75,000 (Junior Data Engineer)  \n- **2019**: $80,000 (Data Engineer)  \n- **2020**: $84,000 (Data Engineer)  \n- **2021**: $95,000 (Senior Data Enginee

In [31]:
def fetch_context(original_question: str, history: list[dict] = []) -> list[Document]:
    """
    Re-write the original question using llm to retrieve more relevant content from vector database.
    Fetch relevant content for both original and rewritten question.
    Merge the relevant content after removing duplicate.
    Rank the content chunks from most relevant to least relevant using llm.

    Args:
        original_question (str): Question of the user
        history (list[dict]): Conversation history

    Returns:
        list[Document]: List of chunks in order of most relevant to least relevant 
    """

    rewritten_question = rewrite_query(question=original_question, history=history)
    chunks_original = fetch_content_unranked(question=original_question)
    chunks_rewritten = fetch_content_unranked(question=rewritten_question)
    chunks_merged = merge_chunks(chunks_original=chunks_original, chunks_rewritten=chunks_rewritten)
    chunks_reranked = rerank(question=original_question, chunks=chunks_merged)

    return chunks_reranked[:FINAL_K]

In [32]:
chunks = fetch_context(original_question=original_question, history=[])

In [33]:
chunks

[Document(id='19dbb1bc-5bb4-4842-b4f4-276922aad7e9', metadata={'source': 'knowledge-base\\employees\\Maxine Thompson.md', 'doc_type': 'employees'}, page_content='- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.'),
 Document(id='02d3d5f4-de51-403c-8a26-3f6dcba4cc1c', metadata={'source': 'knowledge-base\\employees\\Maxine Thompson.md', 'doc_type': 'employees'}, page_content='## Compensation History\n- **2017**: $70,000 (Junior Data Engineer)  \n- **2018**: $75,000 (Junior Data Engineer)  \n- **2019**: $80,000 (Data Engineer)  \n- **2020**: $84,000 (Data Engineer)  \n- **2021**: $95,000 (Senior Data Enginee

In [34]:
def make_rag_messages(question: str, history: list[dict], chunks: list[Document]) -> list[dict]:
    """
    Prepare the messages for final llm call to get the answer of the question

    Args:
        question (str): Question of the user
        history (list[dict]): Conversation history
        chunks (list[Document]): Relevant content chunks

    Returns:
        list[dict]: Messages to call the llm
    """

    context = "\n\n".join(chunk.page_content for chunk in chunks)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(history)
    messages.append(HumanMessage(content=question))

    return messages

In [35]:
messages = make_rag_messages(question=original_question, history=[], chunks=chunks)

In [36]:
messages

[SystemMessage(content='\nYou are a knowledgeable, friendly assistant representing the company Insurellm.\nYou are chatting with a user about Insurellm.\nYour answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.\nIf you don\'t know the answer, say so.\nFor context, here are specific extracts from the Knowledge Base that might be directly relevant to the user\'s question:\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.\n\n## Compensation History\n- **2017**: $70,000 (Junior Data Engineer)  \n- **2018**: $75,000 (Junior Data En

In [37]:
llm = ChatOpenAI(temperature=0, model=MODEL_LLM)
response = llm.invoke(input=messages)

In [38]:
response.content

'Maxine. She was named Insurellm Innovator of the Year in 2023 and received the IIOTY 2023 award.'